In [2]:
!pip install -q -U bitsandbytes
!pip install -q -U transformers
!pip install -q -U xformers
!pip install -q -U peft
!pip install -q -U accelerate
!pip install -q -U datasets
!pip install -q -U trl
!pip install -q -U einops
!pip install -q -U nvidia-ml-py3
!pip install -q -U huggingface_hub

In [68]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: write).
The token `adam` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when push

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [70]:
base_model = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(base_model,
                                             device_map="auto",
                                             torch_dtype=torch.bfloat16)

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [71]:
messages = [
    {"role": "user", "content": "Below is an instruction that describes a task.Write a response that appropriately completes the request. Instruction : List all the cities in a decreasing order of each city's stations' highest latitude. Database Schema: CREATE TABLE station (city VARCHAR, lat INTEGER) Response:"}]

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

generation_args = {
    "max_new_tokens": 200,
    "return_full_text": False,
    "temperature": 0.0,
}

In [72]:
output = pipe(messages, **generation_args)
print(output[0]['generated_text'])

ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

In [80]:
import re

In [81]:
#prompt
input_text = "Below is an instruction that describes a task.Write a response that appropriately completes the request. Instruction : List all the cities in a decreasing order of each city's stations' highest latitude. Database Schema: CREATE TABLE station (city VARCHAR, lat INTEGER) Response:"
input_text = "I am Anushka and code"


#Tokenizes the input text using the loaded tokenizer
#And returns the tokenized input in PyTorch tensor format
input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)

#Generates text based on the tokenized input
outputs = model.generate(**input_ids)

#Decodes the generated output tokens into human-readable text
print(tokenizer.decode(outputs[0]))

decoded_output = tokenizer.decode(outputs[0], skip_special_token=True)
print(decoded_output)

# Remove specific HTML-like tags
cleaned_output = re.sub(r'<.*?>', '', decoded_output)
print(cleaned_output)


/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1375: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


<bos>I am Anushka and code with passion. I am a software engineer and a full stack developer
<bos>I am Anushka and code with passion. I am a software engineer and a full stack developer
I am Anushka and code with passion. I am a software engineer and a full stack developer
